# آموزش تخمین عمق تک‌چشمی با Depth Anything V1

### Notebook آموزشی فارسی — معماری، داده، Fine-tuning، ارزیابی و مشاهدهٔ خروجی

این Notebook بر اساس مخزن رسمی LiheYoung/Depth-Anything (نسخهٔ اول، CVPR 2024) طراحی شده است و برای اجرا در Google Colab مناسب است.

**خروجی‌های یادگیری**

- درک مسئلهٔ Monocular Depth Estimation و تفاوت عمق نسبی و متریک
- شناخت معماری DINOv2 + DPT در Depth Anything
- اجرای مدل از پیش‌آموزش‌دیده و ثبت baseline
- Fine-tuning روی جفت‌های RGB–Depth از NYU Depth V2 یا دیتاست شخصی
- محاسبهٔ AbsRel، RMSE و δ1 و مقایسهٔ قبل/بعد
- مشاهده و ذخیرهٔ نقشهٔ عمق

> پیشنهاد سخت‌افزار: Colab GPU با حداقل 12GB VRAM. پیکربندی پیش‌فرض سبک و آموزشی است.

## 1) دقیقاً چه چیزی را آموزش می‌دهیم؟

از یک تصویر RGB، برای هر پیکسل یک مقدار عمق پیش‌بینی می‌کنیم. این مسئله ذاتاً ill-posed است: چند صحنهٔ سه‌بعدی متفاوت می‌توانند تصویر دوبعدی مشابهی بسازند. مدل باید از نشانه‌هایی مانند اندازهٔ نسبی، پرسپکتیو، انسداد، بافت، سایه و دانش معنایی اشیا استفاده کند.

| نوع خروجی | معنی | واحد | کاربرد |
|---|---|---|---|
| Relative depth | ترتیب و ساختار نزدیک/دور | بدون واحد و تا یک scale/shift نامعین | فهم هندسه، افکت‌ها، ControlNet، پیش‌آموزش |
| Metric depth | فاصلهٔ تقریبی تا دوربین | متر | رباتیک، اندازه‌گیری، ناوبری و 3D |

مدل پایهٔ این مخزن عمدتاً **عمق نسبی** می‌دهد. مسیر metric_depth در مخزن، مدل را با اطلاعات متریک NYU Depth V2 یا KITTI و هد ZoeDepth سازگار می‌کند.

در این Notebook یک Fine-tuning آموزشی و قابل‌فهم انجام می‌دهیم: مدل خروجی inverse-depth نسبی را روی NYU یاد می‌گیرد و هنگام ارزیابی، پیش‌بینی با Ground Truth به‌صورت scale-and-shift هم‌تراز می‌شود. این کار بازتولید کامل pretraining روی 62M تصویر یا کل مسیر ZoeDepth نیست.

## 2) معماری Depth Anything V1

**مسیر رو به جلو**

RGB → نرمال‌سازی و اندازهٔ مضرب 14 → Patchهای 14×14 → انکودر DINOv2 ViT → چهار feature میانی → هد DPT → fusion از عمیق به کم‌عمق → upsampling → نقشهٔ تک‌کانالهٔ عمق نسبی

### اجزای اصلی

1. **DINOv2 encoder**: تصویر را به patch token تبدیل می‌کند. نسخه‌های vits، vitb و vitl به‌ترتیب کوچک، پایه و بزرگ هستند.
2. **Intermediate features**: از چهار بلوک ترنسفورمر feature گرفته می‌شود؛ بنابراین هم اطلاعات معنایی عمیق و هم جزئیات مکانی در دسترس decoder است.
3. **DPT head**: tokenها دوباره به grid دوبعدی تبدیل می‌شوند. کانال هر سطح با convolution یک‌در‌یک تنظیم و رزولوشن‌ها با 4×، 2×، identity و 1/2 ساخته می‌شوند.
4. **Feature fusion blocks**: ویژگی‌ها از عمیق‌ترین سطح به سطوح کم‌عمق‌تر افزوده می‌شوند؛ Residual Convolution و bilinear upsampling مرزها و ساختار چندمقیاسی را بازسازی می‌کنند.
5. **Output head**: convolution نهایی، interpolation تا اندازهٔ ورودی و ReLU یک نقشهٔ مثبت تک‌کاناله تولید می‌کنند.

| Encoder | پارامتر کل اعلام‌شده در مخزن | کاربرد پیشنهادی |
|---|---:|---|
| ViT-S/14 | 24.8M | Colab، آموزش کلاسی و inference سریع |
| ViT-B/14 | 97.5M | تعادل کیفیت و هزینه |
| ViT-L/14 | 335.3M | کیفیت بالاتر و GPU قوی |

نکته: عدد 14 اندازهٔ patch است؛ به همین دلیل ابعاد ورودی بهتر است مضربی از 14 باشد.

## 3) ایدهٔ آموزش در مقالهٔ اصلی

- ابتدا از حدود 1.5M تصویر دارای برچسب برای ساخت یک مدل/معلم اولیه استفاده شد.
- یک data engine برای حدود 62M تصویر بدون برچسب، pseudo-depth تولید کرد.
- دانش‌آموز روی تصویرهای دارای برچسب و pseudo-labelها آموزش دید؛ perturbationهای قوی، مسئله را برای دانش‌آموز دشوارتر کردند تا صرفاً معلم را حفظ نکند.
- auxiliary semantic supervision کمک کرد featureهای معنایی غنی انکودر از دست نروند.
- برای عمق متریک، مدل پایه روی NYU Depth V2 (داخل ساختمان) یا KITTI (رانندگی) Fine-tune شد.

این مقیاس در یک کلاس یا Colab قابل بازتولید نیست. هدف این Notebook فهم مسیر مهندسی و اجرای یک Fine-tuning کوچک اما واقعی است.

## 4) نصب و دریافت مخزن رسمی

Runtime را روی GPU قرار دهید: Runtime → Change runtime type → T4 GPU.

In [ ]:
!test -d /content/Depth-Anything || git clone --depth 1 https://github.com/LiheYoung/Depth-Anything.git /content/Depth-Anything
!git -C /content/Depth-Anything checkout 1d03336771fe09c5398ffdd211441e33941a97dc
%cd /content/Depth-Anything
!pip -q install -r requirements.txt datasets h5py matplotlib tqdm


In [ ]:
import os
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import ColorJitter
from tqdm.auto import tqdm

from depth_anything.dpt import DepthAnything

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


## 5) تنظیمات آزمایش

برای اجرای سریع کلاسی، فقط 64 نمونهٔ train و 16 نمونهٔ validation می‌گیریم و انکودر را freeze می‌کنیم. برای پروژهٔ واقعی تعداد نمونه‌ها و epochها را افزایش دهید.

اگر خطای کمبود حافظه گرفتید: IMG_SIZE را 224، BATCH_SIZE را 1 و TRAIN_SAMPLES را 32 کنید.

In [ ]:
# ---------- انتخاب داده ----------
DATA_SOURCE = 'hf_nyu'       # 'hf_nyu' یا 'custom_folder'
HF_DATASET_ID = 'sayakpaul/nyu_depth_v2'

# فقط در حالت custom_folder استفاده می‌شود.
CUSTOM_ROOT = '/content/my_depth_dataset'
CUSTOM_DEPTH_SCALE = 1000.0  # اگر PNG عمق بر حسب میلی‌متر است؛ برای متر برابر 1.0

# ---------- مدل و آموزش ----------
ENCODER = 'vits'             # vits / vitb / vitl
IMG_SIZE = 280               # مضرب 14
TRAIN_SAMPLES = 64
VAL_SAMPLES = 16
BATCH_SIZE = 2
EPOCHS = 2
FREEZE_ENCODER = True
HEAD_LR = 1e-4
ENCODER_LR = 1e-5
WEIGHT_DECAY = 1e-2
MIN_DEPTH_M = 0.1
MAX_DEPTH_M = 10.0           # دامنهٔ رایج NYU داخل ساختمان
NUM_WORKERS = 2

assert IMG_SIZE % 14 == 0, 'IMG_SIZE باید مضربی از 14 باشد.'
print('Patch grid:', IMG_SIZE // 14, 'x', IMG_SIZE // 14)


## 6) دریافت زیرمجموعهٔ NYU Depth V2 یا دیتاست شخصی

حالت hf_nyu داده را streaming می‌خواند و فقط تعداد نمونهٔ مشخص‌شده را materialize می‌کند؛ بنابراین قرار نیست کل دیتاست دانلود شود.

برای دیتاست شخصی ساختار زیر را بسازید و DATA_SOURCE را به custom_folder تغییر دهید:

my_depth_dataset / train / images

my_depth_dataset / train / depth

my_depth_dataset / val / images

my_depth_dataset / val / depth

نام stem جفت‌ها باید یکسان باشد؛ مثلاً images/0001.jpg و depth/0001.png.

In [ ]:
def load_hf_subset(split, count, seed):
    from datasets import load_dataset
    try:
        stream = load_dataset(HF_DATASET_ID, split=split, streaming=True)
        stream = stream.shuffle(seed=seed, buffer_size=max(256, count * 4))
        records = list(stream.take(count))
    except Exception as error:
        raise RuntimeError(
            'خواندن streaming دیتاست NYU ناموفق بود. اینترنت Colab را بررسی کنید یا '
            'DATA_SOURCE را روی custom_folder قرار دهید.'
        ) from error
    if not records:
        raise RuntimeError(f'هیچ نمونه‌ای از split={split} دریافت نشد.')
    return records


def collect_custom_pairs(root, split):
    root = Path(root)
    image_dir = root / split / 'images'
    depth_dir = root / split / 'depth'
    if not image_dir.exists() or not depth_dir.exists():
        raise FileNotFoundError(f'پوشه‌های {image_dir} و {depth_dir} پیدا نشدند.')

    image_exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    depth_exts = {'.png', '.tif', '.tiff'}
    images = {p.stem: p for p in image_dir.iterdir() if p.suffix.lower() in image_exts}
    depths = {p.stem: p for p in depth_dir.iterdir() if p.suffix.lower() in depth_exts}
    common = sorted(set(images) & set(depths))
    if not common:
        raise RuntimeError(f'هیچ جفت RGB–Depth با stem یکسان در split={split} یافت نشد.')
    return [(str(images[k]), str(depths[k])) for k in common]


if DATA_SOURCE == 'hf_nyu':
    print('Reading a small streamed subset of NYU Depth V2 ...')
    train_records = load_hf_subset('train', TRAIN_SAMPLES, SEED)
    val_records = load_hf_subset('validation', VAL_SAMPLES, SEED + 1)
    depth_scale = 1.0
elif DATA_SOURCE == 'custom_folder':
    train_records = collect_custom_pairs(CUSTOM_ROOT, 'train')[:TRAIN_SAMPLES]
    val_records = collect_custom_pairs(CUSTOM_ROOT, 'val')[:VAL_SAMPLES]
    depth_scale = CUSTOM_DEPTH_SCALE
else:
    raise ValueError('DATA_SOURCE باید hf_nyu یا custom_folder باشد.')

print('Train samples:', len(train_records))
print('Validation samples:', len(val_records))


In [ ]:
def read_rgb_depth(record, depth_scale=1.0):
    if isinstance(record, dict):
        rgb = record['image']
        depth = record['depth_map']
    else:
        rgb = Image.open(record[0])
        depth = Image.open(record[1])

    rgb = ImageOps.exif_transpose(rgb).convert('RGB')
    depth_np = np.asarray(depth).astype(np.float32)
    if depth_np.ndim == 3:
        depth_np = depth_np[..., 0]
    depth_np = depth_np / float(depth_scale)
    return rgb, depth_np


rgb0, depth0 = read_rgb_depth(train_records[0], depth_scale)
valid0 = np.isfinite(depth0) & (depth0 > MIN_DEPTH_M) & (depth0 < MAX_DEPTH_M)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(rgb0)
axes[0].set_title('RGB')
im = axes[1].imshow(np.where(valid0, depth0, np.nan), cmap='turbo_r', vmin=MIN_DEPTH_M, vmax=MAX_DEPTH_M)
axes[1].set_title('Ground-truth depth (meter)')
for ax in axes:
    ax.axis('off')
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.show()

print('RGB size:', rgb0.size)
print('Depth shape:', depth0.shape)
print('Valid depth min/max (m):', float(depth0[valid0].min()), float(depth0[valid0].max()))


## 7) Dataset و تبدیل‌های هم‌زمان RGB–Depth

هر تبدیل هندسی باید دقیقاً روی RGB، depth و mask یکسان اعمال شود. ColorJitter فقط روی RGB اعمال می‌شود. target آموزشی inverse depth است:

inverse_depth = 1 / metric_depth

پس مقدار بزرگ‌تر معمولاً به معنای نزدیک‌تر بودن است.

In [ ]:
class RGBDepthDataset(Dataset):
    def __init__(self, records, train, depth_scale=1.0):
        self.records = records
        self.train = train
        self.depth_scale = depth_scale
        self.jitter = ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        rgb, depth_np = read_rgb_depth(self.records[index], self.depth_scale)
        rgb_t = TF.to_tensor(rgb)
        depth_t = torch.from_numpy(depth_np).unsqueeze(0)
        mask_t = torch.isfinite(depth_t) & (depth_t > MIN_DEPTH_M) & (depth_t < MAX_DEPTH_M)
        depth_t = torch.where(mask_t, depth_t, torch.zeros_like(depth_t))

        rgb_t = TF.resize(rgb_t, [IMG_SIZE, IMG_SIZE], antialias=True)
        depth_t = F.interpolate(depth_t.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze(0)
        mask_t = F.interpolate(mask_t.float().unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode='nearest').squeeze(0).bool()

        if self.train and random.random() < 0.5:
            rgb_t = torch.flip(rgb_t, dims=[2])
            depth_t = torch.flip(depth_t, dims=[2])
            mask_t = torch.flip(mask_t, dims=[2])

        if self.train:
            rgb_t = self.jitter(rgb_t).clamp(0, 1)

        rgb_t = TF.normalize(rgb_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        inv_depth_t = torch.where(mask_t, 1.0 / depth_t.clamp_min(MIN_DEPTH_M), torch.zeros_like(depth_t))

        return {
            'image': rgb_t,
            'depth': depth_t,
            'inv_depth': inv_depth_t,
            'mask': mask_t,
        }


train_dataset = RGBDepthDataset(train_records, train=True, depth_scale=depth_scale)
val_dataset = RGBDepthDataset(val_records, train=False, depth_scale=depth_scale)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')
)

batch = next(iter(train_loader))
for key, value in batch.items():
    print(f'{key:10s}: {tuple(value.shape)} | {value.dtype}')


## 8) بارگذاری Depth Anything Small و بررسی featureها

در تنظیم پیش‌فرض، encoder منجمد است و فقط depth_head آموزش می‌بیند. این روش سریع‌تر است و با دیتاست کوچک overfit کمتری دارد. برای Fine-tuning کامل، FREEZE_ENCODER=False بگذارید و ترجیحاً داده و epoch بیشتری استفاده کنید.

In [ ]:
MODEL_ID = f'LiheYoung/depth_anything_{ENCODER}14'
model = DepthAnything.from_pretrained(MODEL_ID).to(DEVICE)

for parameter in model.pretrained.parameters():
    parameter.requires_grad = not FREEZE_ENCODER
for parameter in model.depth_head.parameters():
    parameter.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {MODEL_ID}')
print(f'Total parameters: {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')


In [ ]:
# مشاهدهٔ تجربی tokenهای چهار لایهٔ میانی encoder
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.inference_mode():
    features = model.pretrained.get_intermediate_layers(dummy, 4, return_class_token=True)

grid = IMG_SIZE // 14
print('Expected patch grid:', grid, 'x', grid, '=', grid * grid, 'tokens')
for i, (patch_tokens, cls_token) in enumerate(features, 1):
    print(f'Feature {i}: patch tokens={tuple(patch_tokens.shape)}, cls token={tuple(cls_token.shape)}')
del dummy, features
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()


## 9) Loss ناوردا به Scale و Shift

خروجی مدل نسبی است؛ بنابراین مقایسهٔ مستقیم آن با متر منصفانه نیست. برای هر تصویر ضرایب a و b را با least squares پیدا می‌کنیم:

aligned_prediction = a × prediction + b

سپس L1 روی inverse-depth معتبر و یک gradient loss برای حفظ مرزها محاسبه می‌شود. این ایده با ماهیت relative-depth سازگار است.

نکتهٔ آموزشی: اگر هدف شما عمق متریک بدون Ground Truth در زمان inference است، باید مسیر metric_depth/ZoeDepth را آموزش دهید؛ هم‌ترازی این Notebook فقط برای loss و ارزیابی supervised استفاده می‌شود.

In [ ]:
def align_scale_and_shift(prediction, target, mask, eps=1e-6):
    if prediction.ndim == 3:
        prediction = prediction.unsqueeze(1)
    if target.ndim == 3:
        target = target.unsqueeze(1)
    if mask.ndim == 3:
        mask = mask.unsqueeze(1)

    mask_f = mask.float()
    dims = (1, 2, 3)
    a00 = (mask_f * prediction * prediction).sum(dims)
    a01 = (mask_f * prediction).sum(dims)
    a11 = mask_f.sum(dims)
    b0 = (mask_f * prediction * target).sum(dims)
    b1 = (mask_f * target).sum(dims)

    determinant = a00 * a11 - a01 * a01
    safe_det = torch.where(determinant.abs() > eps, determinant, torch.ones_like(determinant))
    scale = (a11 * b0 - a01 * b1) / safe_det
    shift = (-a01 * b0 + a00 * b1) / safe_det
    valid_system = determinant.abs() > eps
    scale = torch.where(valid_system, scale, torch.ones_like(scale))
    shift = torch.where(valid_system, shift, torch.zeros_like(shift))

    aligned = scale[:, None, None, None] * prediction + shift[:, None, None, None]
    return aligned, scale, shift


def masked_mean(values, mask):
    mask_f = mask.float()
    return (values * mask_f).sum() / mask_f.sum().clamp_min(1.0)


def scale_shift_invariant_loss(prediction, target_inv_depth, mask, gradient_weight=0.5):
    aligned, _, _ = align_scale_and_shift(prediction, target_inv_depth, mask)
    data_loss = masked_mean((aligned - target_inv_depth).abs(), mask)

    valid_x = mask[:, :, :, 1:] & mask[:, :, :, :-1]
    valid_y = mask[:, :, 1:, :] & mask[:, :, :-1, :]
    pred_dx = aligned[:, :, :, 1:] - aligned[:, :, :, :-1]
    pred_dy = aligned[:, :, 1:, :] - aligned[:, :, :-1, :]
    target_dx = target_inv_depth[:, :, :, 1:] - target_inv_depth[:, :, :, :-1]
    target_dy = target_inv_depth[:, :, 1:, :] - target_inv_depth[:, :, :-1, :]
    grad_loss = masked_mean((pred_dx - target_dx).abs(), valid_x)
    grad_loss = grad_loss + masked_mean((pred_dy - target_dy).abs(), valid_y)
    return data_loss + gradient_weight * grad_loss


## 10) معیارها

- AbsRel: میانگین خطای مطلق تقسیم بر عمق واقعی؛ کمتر بهتر است.
- RMSE: ریشهٔ میانگین مربع خطا بر حسب متر؛ کمتر بهتر است.
- δ1: سهم پیکسل‌هایی که نسبت pred/gt یا gt/pred کمتر از 1.25 است؛ بیشتر بهتر است.

چون مدل relative است، قبل از تبدیل inverse-depth به متر، scale و shift با GT همان تصویر هم‌تراز می‌شوند. این معیارها کیفیت ساختار نسبی را می‌سنجند و معادل ارزیابی یک مدل metric مستقل نیستند.

In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    absrel_sum = 0.0
    squared_error_sum = 0.0
    delta1_sum = 0.0
    pixel_count = 0

    for batch in loader:
        image = batch['image'].to(DEVICE, non_blocking=True)
        depth = batch['depth'].to(DEVICE, non_blocking=True)
        target_inv = batch['inv_depth'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)

        pred_inv = model(image)
        aligned_inv, _, _ = align_scale_and_shift(pred_inv, target_inv, mask)
        aligned_inv = aligned_inv.clamp(min=1.0 / MAX_DEPTH_M, max=1.0 / MIN_DEPTH_M)
        pred_depth = 1.0 / aligned_inv

        valid_pred = torch.isfinite(pred_depth) & (pred_depth > 0)
        valid = mask & valid_pred
        gt = depth[valid]
        pd = pred_depth[valid]
        count = gt.numel()
        if count == 0:
            continue

        absrel_sum += ((pd - gt).abs() / gt.clamp_min(1e-6)).sum().item()
        squared_error_sum += ((pd - gt) ** 2).sum().item()
        ratio = torch.maximum(pd / gt.clamp_min(1e-6), gt / pd.clamp_min(1e-6))
        delta1_sum += (ratio < 1.25).sum().item()
        pixel_count += count

    return {
        'AbsRel': absrel_sum / max(pixel_count, 1),
        'RMSE': (squared_error_sum / max(pixel_count, 1)) ** 0.5,
        'delta1': delta1_sum / max(pixel_count, 1),
    }


@torch.inference_mode()
def predict_aligned_depth(model, batch):
    model.eval()
    image = batch['image'].to(DEVICE)
    target_inv = batch['inv_depth'].to(DEVICE)
    mask = batch['mask'].to(DEVICE)
    pred_inv = model(image)
    aligned_inv, _, _ = align_scale_and_shift(pred_inv, target_inv, mask)
    aligned_inv = aligned_inv.clamp(min=1.0 / MAX_DEPTH_M, max=1.0 / MIN_DEPTH_M)
    return (1.0 / aligned_inv).cpu()


## 11) Baseline قبل از Fine-tuning

In [ ]:
demo_batch = next(iter(val_loader))
baseline_depth_viz = predict_aligned_depth(model, demo_batch)[0, 0].numpy()
baseline_metrics = evaluate(model, val_loader)
print('Baseline metrics')
for key, value in baseline_metrics.items():
    print(f'{key:7s}: {value:.4f}')


## 12) حلقهٔ آموزش

برای encoder و head نرخ یادگیری جدا داریم. Mixed Precision و gradient clipping برای پایداری و مصرف حافظهٔ کمتر فعال می‌شوند.

In [ ]:
head_parameters = [p for p in model.depth_head.parameters() if p.requires_grad]
encoder_parameters = [p for p in model.pretrained.parameters() if p.requires_grad]
parameter_groups = [{'params': head_parameters, 'lr': HEAD_LR}]
if encoder_parameters:
    parameter_groups.append({'params': encoder_parameters, 'lr': ENCODER_LR})

optimizer = torch.optim.AdamW(parameter_groups, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
use_amp = DEVICE.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

history = {'train_loss': [], 'val_absrel': [], 'val_rmse': [], 'val_delta1': []}
best_rmse = float('inf')
CHECKPOINT_PATH = '/content/depth_anything_v1_finetuned_nyu.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    if FREEZE_ENCODER:
        model.pretrained.eval()

    running_loss = 0.0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in progress:
        image = batch['image'].to(DEVICE, non_blocking=True)
        target_inv = batch['inv_depth'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        amp_context = torch.autocast(device_type='cuda', dtype=torch.float16) if use_amp else nullcontext()
        with amp_context:
            pred_inv = model(image)
            loss = scale_shift_invariant_loss(pred_inv, target_inv, mask)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        progress.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()
    epoch_loss = running_loss / max(len(train_loader), 1)
    metrics = evaluate(model, val_loader)
    history['train_loss'].append(epoch_loss)
    history['val_absrel'].append(metrics['AbsRel'])
    history['val_rmse'].append(metrics['RMSE'])
    history['val_delta1'].append(metrics['delta1'])

    print(
        f'Epoch {epoch}: loss={epoch_loss:.4f} | '
        f'AbsRel={metrics["AbsRel"]:.4f} | RMSE={metrics["RMSE"]:.4f} m | δ1={metrics["delta1"]:.4f}'
    )

    if metrics['RMSE'] < best_rmse:
        best_rmse = metrics['RMSE']
        trainable_state = {
            name: parameter.detach().cpu()
            for name, parameter in model.named_parameters() if parameter.requires_grad
        }
        torch.save({
            'trainable_state': trainable_state,
            'encoder': ENCODER,
            'img_size': IMG_SIZE,
            'data_source': DATA_SOURCE,
            'best_rmse': best_rmse,
        }, CHECKPOINT_PATH)

print('Best checkpoint:', CHECKPOINT_PATH)


In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], marker='o')
axes[0].set_title('Training loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_absrel'], marker='o', label='AbsRel')
axes[1].plot(epochs, history['val_rmse'], marker='s', label='RMSE (m)')
axes[1].plot(epochs, history['val_delta1'], marker='^', label='delta1')
axes[1].set_title('Validation metrics')
axes[1].set_xlabel('Epoch')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.show()


## 13) مقایسهٔ قبل و بعد از آموزش

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
missing, unexpected = model.load_state_dict(checkpoint['trainable_state'], strict=False)
model.to(DEVICE).eval()

finetuned_metrics = evaluate(model, val_loader)
finetuned_depth_viz = predict_aligned_depth(model, demo_batch)[0, 0].numpy()
gt_depth_viz = demo_batch['depth'][0, 0].numpy()
valid_viz = demo_batch['mask'][0, 0].numpy()

print('Metric comparison')
print(f'{"Metric":8s} | {"Before":>10s} | {"After":>10s}')
print('-' * 36)
for key in ['AbsRel', 'RMSE', 'delta1']:
    print(f'{key:8s} | {baseline_metrics[key]:10.4f} | {finetuned_metrics[key]:10.4f}')

error_map = np.abs(finetuned_depth_viz - gt_depth_viz)
error_map[~valid_viz] = np.nan
gt_show = np.where(valid_viz, gt_depth_viz, np.nan)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
images = [gt_show, baseline_depth_viz, finetuned_depth_viz, error_map]
titles = ['Ground truth (m)', 'Before fine-tuning', 'After fine-tuning', 'Absolute error (m)']
cmaps = ['turbo_r', 'turbo_r', 'turbo_r', 'magma']

for i, (ax, array, title, cmap) in enumerate(zip(axes, images, titles, cmaps)):
    if i < 3:
        im = ax.imshow(array, cmap=cmap, vmin=MIN_DEPTH_M, vmax=MAX_DEPTH_M)
    else:
        im = ax.imshow(array, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## 14) Inference روی یک تصویر دلخواه بدون Ground Truth

در این حالت فقط relative depth قابل گزارش است؛ عدد پیکسل متر نیست. خروجی را با percentile نرمال می‌کنیم تا تصویر قابل مشاهده باشد. مقدار روشن‌تر در نقشهٔ gray معمولاً امتیاز inverse-depth بیشتر و ناحیهٔ نزدیک‌تر را نشان می‌دهد.

In [ ]:
def preprocess_rgb_only(pil_image):
    rgb = ImageOps.exif_transpose(pil_image).convert('RGB')
    tensor = TF.to_tensor(rgb)
    tensor = TF.resize(tensor, [IMG_SIZE, IMG_SIZE], antialias=True)
    tensor = TF.normalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return rgb, tensor.unsqueeze(0)


@torch.inference_mode()
def predict_relative_depth(pil_image):
    rgb, tensor = preprocess_rgb_only(pil_image)
    pred = model(tensor.to(DEVICE))
    pred = F.interpolate(pred.unsqueeze(1), size=(rgb.height, rgb.width), mode='bilinear', align_corners=False)[0, 0]
    pred = pred.cpu().numpy()
    low, high = np.percentile(pred, [2, 98])
    normalized = np.clip((pred - low) / max(high - low, 1e-6), 0, 1)
    return rgb, pred, normalized


# برای تصویر خودتان مسیر را وارد کنید؛ در غیر این صورت یک نمونهٔ validation نمایش داده می‌شود.
TEST_IMAGE_PATH = None  # مثال: '/content/my_photo.jpg'

if TEST_IMAGE_PATH:
    test_image = Image.open(TEST_IMAGE_PATH)
else:
    test_image, _ = read_rgb_depth(val_records[0], depth_scale)

rgb_test, raw_relative, relative_vis = predict_relative_depth(test_image)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(rgb_test)
axes[0].set_title('Input image')
im = axes[1].imshow(relative_vis, cmap='inferno')
axes[1].set_title('Relative inverse-depth (normalized)')
for ax in axes:
    ax.axis('off')
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.show()


In [ ]:
# ذخیرهٔ نقشهٔ نمایشی و checkpoint
OUTPUT_DEPTH_PNG = '/content/predicted_relative_depth.png'
Image.fromarray((relative_vis * 255).astype(np.uint8)).save(OUTPUT_DEPTH_PNG)
print('Depth image:', OUTPUT_DEPTH_PNG)
print('Checkpoint:', CHECKPOINT_PATH)

# در Colab، در صورت نیاز خطوط زیر را اجرا کنید:
# from google.colab import files
# files.download(OUTPUT_DEPTH_PNG)
# files.download(CHECKPOINT_PATH)


## 15) چگونه نتایج را درست تفسیر کنیم؟

1. با فقط 64 تصویر و 2 epoch انتظار SOTA نداریم؛ هدف مشاهدهٔ pipeline است.
2. کاهش train loss بدون بهبود validation نشانهٔ overfitting است.
3. نواحی شفاف، آینه، پنجره، سطح براق، تاریکی و مرزهای باریک معمولاً دشوارند.
4. colormap فاصله نیست؛ colorbar و واحد را همیشه مشخص کنید.
5. relative output را بدون calibration یا metric head به متر تبدیل نکنید.
6. split تصادفی فریم‌های یک ویدئو ممکن است leakage ایجاد کند؛ split واقعی باید scene-level باشد.
7. برای ارزیابی جدی از crop و پروتکل استاندارد همان benchmark استفاده کنید.

## 16) پیشنهاد تدریس معماری در کلاس

**مرحلهٔ اول — شهود:** یک تصویر را نشان دهید و بپرسید مدل بدون stereo یا LiDAR از کجا نزدیک/دور را می‌فهمد.

**مرحلهٔ دوم — tensorها:** سلول feature inspection را اجرا کنید و تعداد patch tokenها را از (IMG_SIZE/14)² محاسبه کنید.

**مرحلهٔ سوم — encoder/decoder:** توضیح دهید DINOv2 چه چیزی را می‌فهمد و DPT چگونه tokenهای کم‌رزولوشن را به نقشهٔ dense تبدیل می‌کند.

**مرحلهٔ چهارم — ambiguity:** یک relative map را نشان دهید و از دانشجو بخواهید توضیح دهد چرا مستقیماً متر نیست.

**مرحلهٔ پنجم — آزمایش:** baseline را ثبت، فقط head را Fine-tune و تغییر معیارها و مرزها را تحلیل کنید.

**پرسش کلیدی:** چرا برای depth estimation فقط feature نهایی ViT کافی نیست و decoder از چهار لایهٔ میانی استفاده می‌کند؟

## 17) تمرین‌های پیشنهادی

### ساده

- IMG_SIZE را از 280 به 224 تغییر دهید؛ زمان و کیفیت را مقایسه کنید.
- ColorJitter را خاموش کنید و منحنی loss را بررسی کنید.
- روی سه تصویر شخصی، failure caseها را علامت‌گذاری کنید.

### متوسط

- FREEZE_ENCODER=False و LR انکودر را ده برابر کوچک‌تر از head قرار دهید.
- تعداد نمونه‌ها را به 256 و epoch را به 5 افزایش دهید.
- gradient_weight را بین 0، 0.25، 0.5 و 1 مقایسه کنید.

### سخت

- به‌جای square resize، resize با حفظ aspect ratio و padding مضرب 14 پیاده‌سازی کنید.
- SILog loss و multi-scale gradient loss را اضافه و با SSI مقایسه کنید.
- مسیر رسمی metric_depth مبتنی بر ZoeDepth را روی NYU اجرا کنید و خروجی متریک مستقل از GT بسازید.
- domain shift را با آموزش روی NYU و تست روی تصاویر outdoor تحلیل کنید.

## 18) منابع اصلی

- Repository: https://github.com/LiheYoung/Depth-Anything
- Snapshot used in this Notebook: commit 1d03336771fe09c5398ffdd211441e33941a97dc
- Paper: https://arxiv.org/abs/2401.10891
- Metric-depth path: https://github.com/LiheYoung/Depth-Anything/tree/main/metric_depth
- NYU Depth V2 dataset card: https://huggingface.co/datasets/sayakpaul/nyu_depth_v2
- جدیدتر از این ریپو: https://github.com/DepthAnything/Depth-Anything-V2

**نکتهٔ نسخه:** این Notebook عمداً V1 را مطابق لینک درخواست‌شده تدریس می‌کند. برای پروژهٔ جدید، V2 را نیز جداگانه مقایسه کنید.